# CAR-bench Evaluation: Qwen2.5-Coder-7B-Instruct Baseline

This notebook guides you through running a local baseline evaluation of the raw (unaligned) **Qwen2.5-Coder-7B-Instruct** model on the CAR-bench dataset hosted on Hugging Face at [johanneskirmayr/car-bench-dataset](https://huggingface.co/datasets/johanneskirmayr/car-bench-dataset).

It is designed to run in cloud environments like Google Colab or Kaggle with a single GPU (such as a T4). To achieve this efficiently and prevent Out-Of-Memory (OOM) errors, it serves the model locally using `vLLM` in 4-bit AWQ precision while delegating the simulated user and policy evaluator roles to a commercial frontier API (e.g. Gemini 2.5 Flash).

**Dataset Download Note**: The `car_bench` simulation environment automatically retrieves the benchmark tasks and mock context databases directly from Hugging Face on the first execution run.

## Pipeline Structure
1. **Environment Setup**: Clone repository, install packages, and set up `third_party/car-bench`.
2. **Risk Management & Persistent Storage**: Configure Google Drive/Kaggle backups and memory purges.
3. **Local vLLM Inference Server**: Spawn a background vLLM instance serving `Qwen/Qwen2.5-Coder-7B-Instruct-AWQ`.
4. **Scenario Configuration**: Generate a TOML scenario file and set environment variables.
5. **Execution**: Run the orchestrator pipeline to evaluate the model on the benchmark splits.
6. **Result Visualizations & Comparison**: Plot performance metrics and compare against frontier baselines.

## 1. Environment Setup

In [ ]:
# Check if running in Google Colab or Kaggle
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Clone repository if running in a clean cloud VM
import os
if IN_COLAB or not os.path.exists("src"):
    print("Cloning repository to retrieve wiki.md policy and scripts...")
    !git clone --recursive https://github.com/CAR-bench/car-bench-ijcai.git
    %cd car-bench-ijcai
else:
    print("Running in local workspace.")

# Robust check/fix for submodule/dependency content in third_party/car-bench
if not os.path.exists("third_party/car-bench/pyproject.toml"):
    print("Submodule files missing in third_party/car-bench. Attempting to restore...")
    !git submodule update --init --recursive
    if not os.path.exists("third_party/car-bench/pyproject.toml"):
        print("Directly cloning official car-bench dependency...")
        import shutil
        if os.path.exists("third_party/car-bench"):
            try:
                shutil.rmtree("third_party/car-bench")
            except Exception:
                !rm -rf third_party/car-bench
        !git clone --depth 1 https://github.com/CAR-bench/car-bench.git third_party/car-bench


In [ ]:
# Install uv package manager and dependencies
!pip install -q uv
!uv pip install --system -q a2a-sdk[http-server]>=1.0.0 httpx loguru pydantic python-dotenv uvicorn nest-asyncio matplotlib pandas seaborn psutil
!uv pip install --system -e third_party/car-bench
!uv pip install --system -q vllm

## 2. Risk Management & Persistent Storage

To ensure the evaluation runs reliably without data loss or OOM crashes, we establish:
1. **Google Drive Mount**: Persistently saves outputs to survive runtime timeouts.
2. **Process Port Cleanup**: Terminates orphaned background servers to prevent "Address already in use" errors on re-runs.
3. **Memory Purge**: Routinely sweeps PyTorch's cache.

In [ ]:
# Mount Google Drive if in Colab to preserve final evaluation files
PERSISTENT_DIR = "./output"
if IN_COLAB:
    try:
        from google.colab import drive
        print("Mounting Google Drive for persistent artifact backups...")
        drive.mount('/content/drive')
        PERSISTENT_DIR = "/content/drive/MyDrive/car_bench_eval_baseline"
    except Exception as e:
        print(f"Google Drive mount skipped or failed: {e}. Output will be saved locally.")
elif os.path.exists("/kaggle/working"):
    PERSISTENT_DIR = "/kaggle/working/output"

os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target persistent output directory: {PERSISTENT_DIR}")

In [ ]:
import psutil
import torch
import gc

def cleanup_ports(ports=[8000, 8080, 8081]):
    """Kills any active local processes listening on specified ports to prevent conflicts."""
    print(f"Scanning for active processes on ports: {ports}...")
    cleaned = 0
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            connections = proc.connections()
            for conn in connections:
                if conn.laddr.port in ports:
                    print(f"Killing process {proc.info['name']} (PID: {proc.pid}) listening on port {conn.laddr.port}...")
                    proc.kill()
                    cleaned += 1
                    break
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    print(f"Port cleanup finished. Terminated {cleaned} process(es).")

def clear_memory():
    """Performs garbage collection and empties PyTorch's CUDA cache to prevent Out-Of-Memory errors."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Purged PyTorch GPU memory cache.")
    else:
        print("CUDA is not available. Skipped GPU memory cache purge.")

# Perform initial cleanup
cleanup_ports()
clear_memory()

## 3. Dataset Download & Exploration

We download the official CAR-bench dataset from Hugging Face at [johanneskirmayr/car-bench-dataset](https://huggingface.co/datasets/johanneskirmayr/car-bench-dataset) to verify and cache the training/validation data splits locally.


In [ ]:
from datasets import load_dataset
import os

print("Downloading CAR-bench dataset from Hugging Face (johanneskirmayr/car-bench-dataset)...")
configs = ["tasks_base", "tasks_disambiguation", "tasks_hallucination"]
splits = ["train", "test"]

os.makedirs("data", exist_ok=True)

for config in configs:
    for split in splits:
        print(f"Downloading split: config={config}, split={split}...")
        ds = load_dataset("johanneskirmayr/car-bench-dataset", config, split=split)
        print(f"  Successfully loaded {len(ds)} samples.")
        
        # Save raw dataset copy locally for verification and exploration
        raw_path = f"data/raw_{config}_{split}.jsonl"
        ds.to_json(raw_path)
        print(f"  Saved locally to: {raw_path}")


## 4. Local vLLM Inference Server

We launch a background `vLLM` instance. Running 7B models in full precision can exceed a T4 GPU's 15GB VRAM limits during inference. We utilize `Qwen/Qwen2.5-Coder-7B-Instruct-AWQ` which fits inside ~6GB VRAM and leaves the rest for task execution.

In [ ]:
import subprocess
import time
import httpx

# Terminate port 8000 in case of previous runs
cleanup_ports([8000])

print("Launching vLLM background server with Qwen2.5-Coder-7B-Instruct-AWQ...")
vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
    "--port", "8000",
    "--gpu-memory-utilization", "0.80",
    "--max-model-len", "8000"
]

# Open log file to keep notebook cells readable
log_file_path = "vllm_server.log"
vllm_log = open(log_file_path, "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=vllm_log)

# Poll model API until it becomes ready
is_ready = False
print("Waiting for vLLM server to start (monitoring health endpoint)...")
for attempt in range(40):
    try:
        response = httpx.get("http://localhost:8000/v1/models", timeout=2)
        if response.status_code == 200:
            print("\nvLLM server successfully started and is ready!")
            is_ready = True
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(10)

if not is_ready:
    print("\n[ERROR] vLLM server failed to start within the timeout period.")
    if os.path.exists(log_file_path):
        print("\n--- Last 30 lines of vLLM Logs ---")
        with open(log_file_path, "r") as f:
            lines = f.readlines()
            print("".join(lines[-30:]))
    raise RuntimeError("vLLM startup failed. Please inspect the log file above.")

## 5. Scenario Configuration

We create `scenarios/track_1_agent_under_test/qwen_coder_baseline.toml` defining the evaluator, agent, and test configuration.

For the baseline run, we test **5 tasks per split** (Base, Hallucination, Disambiguation) to evaluate performance across all areas quickly. We also prompt the user to input their commercial API key for the driver simulator (User Simulator) and policy checker (Policy Evaluator).

In [ ]:
import getpass

print("Enter your Gemini API Key (used for simulating user/evaluating policies):")
gemini_key = getpass.getpass()

# Write env configuration to .env file
with open(".env", "w") as f:
    f.write(f"GEMINI_API_KEY={gemini_key}\n")
    # Redirect LiteLLM OpenAI requests for the Agent to the local vLLM server
    f.write("OPENAI_API_BASE=http://localhost:8000/v1\n")
    f.write("OPENAI_API_KEY=local-dummy-key\n")

print("Created .env configuration file successfully.")

In [ ]:
import os

scenario_content = """# CAR-bench evaluation scenario for raw Qwen2.5-Coder-7B-Instruct
[evaluator]
endpoint = "http://127.0.0.1:8081"
cmd = "python src/evaluator/server.py --host 127.0.0.1 --port 8081"

[agent_under_test]
endpoint = "http://127.0.0.1:8080"
cmd = "python src/track_1_agent_under_test/server.py --host 127.0.0.1 --port 8080 --agent-llm openai/Qwen/Qwen2.5-Coder-7B-Instruct-AWQ --temperature 0.0"

[config]
num_trials = 1
task_split = "train"
tasks_base_num_tasks = 5
tasks_hallucination_num_tasks = 5
tasks_disambiguation_num_tasks = 5
max_steps = 30

# Simulated user and policy evaluator use Google Gemini
user_model = "gemini/gemini-2.5-flash"
user_provider = "gemini"
policy_evaluator_model = "gemini/gemini-2.5-flash"
policy_evaluator_provider = "gemini"
"""

os.makedirs("scenarios/track_1_agent_under_test", exist_ok=True)
with open("scenarios/track_1_agent_under_test/qwen_coder_baseline.toml", "w") as f:
    f.write(scenario_content)

print("Created scenarios/track_1_agent_under_test/qwen_coder_baseline.toml")

## 6. Execution

Now we run the benchmark using the `car-bench-run` orchestration tool. It will automatically start the agent and evaluator servers on ports 8080 and 8081, execute the test scenarios sequentially, collect metrics, and shut them down cleanly.

In [ ]:
# Create output directory
os.makedirs("output", exist_ok=True)

# Run the benchmark scenario
!uv run car-bench-run scenarios/track_1_agent_under_test/qwen_coder_baseline.toml --output output/qwen_coder_baseline.json --show-logs

# Backup results to persistent storage
import shutil
if PERSISTENT_DIR != "./output":
    print(f"Backing up results to Drive storage: {PERSISTENT_DIR}...")
    if os.path.exists("output/qwen_coder_baseline.json"):
        shutil.copy("output/qwen_coder_baseline.json", os.path.join(PERSISTENT_DIR, "qwen_coder_baseline.json"))
        print("Backup complete!")

## 7. Result Visualizations & Comparison

We extract evaluation scores from the output JSON and render comparative visualizations against official CAR-bench baselines.

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display, HTML

# Load execution results
result_file = "output/qwen_coder_baseline.json"
if not os.path.exists(result_file) and os.path.exists(os.path.join(PERSISTENT_DIR, "qwen_coder_baseline.json")):
    result_file = os.path.join(PERSISTENT_DIR, "qwen_coder_baseline.json")

try:
    with open(result_file, "r") as f:
        data = json.load(f)

    final_result = data.get("final_result", {})
    pass_rate = final_result.get("pass_rate", 0.0)
    rewards_by_split = final_result.get("pass_power_k_scores_by_split", {})

    # Extract split-level scores
    splits = ["base", "hallucination", "disambiguation"]
    split_scores = []
    for split in splits:
        split_score = rewards_by_split.get(split, {}).get("Pass^1", 0.0) * 100
        split_scores.append(split_score)

    print(f"Overall Pass Rate: {pass_rate:.1f}%")
    print(f"Base Split Pass Rate: {split_scores[0]:.1f}%")
    print(f"Hallucination Split Pass Rate: {split_scores[1]:.1f}%")
    print(f"Disambiguation Split Pass Rate: {split_scores[2]:.1f}%")

    # Save visualization to persistent folder
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))
    categories = ["Overall", "Base Split", "Hallucination Split", "Disambiguation Split"]
    values = [pass_rate] + split_scores

    ax = sns.barplot(x=categories, y=values, palette="viridis")
    plt.title("CAR-bench Baseline Scores (Raw Qwen2.5-Coder-7B-Instruct)", fontsize=14, fontweight="bold", pad=15)
    plt.ylabel("Pass Rate (%)", fontsize=12)
    plt.ylim(0, 100)

    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 1.5),
                    ha='center', va='center', fontsize=11, fontweight="semibold", color='black', xytext=(0, 5),
                    textcoords='offset points')

    plt.tight_layout()
    chart_path = os.path.join(PERSISTENT_DIR, "baseline_scores.png")
    plt.savefig("output/baseline_scores.png", dpi=300)
    plt.savefig(chart_path, dpi=300)
    plt.show()

    # Compare with leaderboard
    comparison_data = {
        "Model": [
            "Claude Opus 4.6", 
            "GPT-5", 
            "Gemini 2.5 Pro", 
            "Qwen2.5-Coder-7B-Instruct (Local Baseline)", 
            "Qwen3-32B", 
            "xLAM-2-32B"
        ],
        "Alignment Method": ["RLHF", "RLHF", "RLHF", "Raw (Unaligned)", "Base SFT/RLHF", "SFT/DPO"],
        "Overall Pass Rate": [0.58, 0.54, 0.38, pass_rate / 100.0, 0.31, 0.16]
    }

    df = pd.DataFrame(comparison_data)
    df["Overall Pass Rate"] = df["Overall Pass Rate"].map(lambda x: f"{x * 100:.1f}%")
    
    print("\n### Performance Comparison Table")
    display(HTML(df.to_html(index=False)))

except Exception as e:
    print(f"Could not load or plot results: {e}. Check if the evaluation run finished successfully.")